In [11]:
import json
from pathlib import Path
import pandas as pd

In [12]:
raw_dir = Path("../data/raw/sec_companyfacts")
out_dir = Path("../data/processed")

In [13]:
info = []
for p in sorted(raw_dir.glob("*_companyfacts.json")):
    d = json.loads(p.read_text(encoding="utf-8"))
    info.append({"ticker": p.name.split("_")[0], "company": d["entityName"], "cik": d["cik"],
                 "us_gaap_tags": len(d["facts"]["us-gaap"]), "size_mb": round(p.stat().st_size / 1e6, 1)})
pd.DataFrame(info)

,ticker,company,cik,us_gaap_tags,size_mb
0,AAPL,Apple Inc.,320193,503,4.2
1,AMZN,AMAZON COM INC,1018724,545,5.0
2,GOOGL,Alphabet Inc.,1652044,543,3.5
3,META,"Meta Platforms, Inc.",1326801,458,3.0
4,MSFT,MICROSOFT CORPORATION,789019,562,5.4
5,NFLX,NETFLIX INC,1065280,490,4.0


In [14]:
tags = [
    "RevenueFromContractWithCustomerExcludingAssessedTax",
    "Revenues",
    "SalesRevenueNet",
]

rows = []
for path in sorted(raw_dir.glob("*_companyfacts.json")):
    facts = json.loads(path.read_text(encoding="utf-8"))["facts"]["us-gaap"]
    for rank, tag in enumerate(tags):
        for r in facts.get(tag, {}).get("units", {}).get("USD", []):
            if r["form"] != "10-K" or "start" not in r:
                continue
            days = (pd.Timestamp(r["end"]) - pd.Timestamp(r["start"])).days
            if 330 <= days <= 400:
                rows.append({"ticker": path.name.split("_")[0], "fiscal_year": int(r["end"][:4]),
                             "period_end": r["end"], "revenue_billion": r["val"] / 1e9,
                             "filed": r["filed"], "tag_rank": rank})

annual = (pd.DataFrame(rows)
          .sort_values(["ticker", "fiscal_year", "tag_rank", "filed"])
          .drop_duplicates(["ticker", "fiscal_year"], keep="first"))
annual = annual.groupby("ticker").tail(10).sort_values(["ticker", "fiscal_year"])

In [15]:
annual.info()

<class 'pandas.core.frame.DataFrame'>
Index: 60 entries, 21 to 278
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   ticker           60 non-null     object 
 1   fiscal_year      60 non-null     int64  
 2   period_end       60 non-null     object 
 3   revenue_billion  60 non-null     float64
 4   filed            60 non-null     object 
 5   tag_rank         60 non-null     int64  
dtypes: float64(1), int64(2), object(3)
memory usage: 3.3+ KB


In [16]:
annual["revenue_billion"].describe().round(2)

count     60.00
mean     213.25
std      164.58
min        8.83
25%       89.20
50%      180.20
75%      288.98
max      716.92
Name: revenue_billion, dtype: float64

In [17]:
raw = pd.DataFrame(rows)
raw["tag"] = raw["tag_rank"].map(dict(enumerate(tags))).str[:20]
raw.groupby(["ticker", "tag"])["fiscal_year"].agg(["min", "max", "nunique"])

min   max  nunique
ticker tag                                      
AAPL   RevenueFromContractW  2017  2025        9
       Revenues              2016  2018        3
       SalesRevenueNet       2007  2017       11
AMZN   RevenueFromContractW  2016  2025       10
       SalesRevenueNet       2007  2017       11
GOOGL  RevenueFromContractW  2016  2024        9
       Revenues              2013  2025       12
META   RevenueFromContractW  2016  2025       10
       Revenues              2010  2017        8
MSFT   RevenueFromContractW  2016  2026       11
       Revenues              2008  2010        3
       SalesRevenueNet       2009  2017        9
NFLX   Revenues              2007  2025       19

In [18]:
pivot = annual.pivot(index="ticker", columns="fiscal_year", values="revenue_billion").round(2)
growth = (pivot.pct_change(axis=1, fill_method=None) * 100).round(2)
growth

fiscal_year,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025,2026
ticker,,,,,,,,,,,
AAPL,NaN,6.30,15.87,-2.04,5.52,33.26,7.79,-2.80,2.02,6.42,NaN
AMZN,NaN,30.80,30.93,20.45,37.62,21.70,9.40,11.83,10.99,12.38,NaN
GOOGL,NaN,22.81,23.42,18.30,12.77,41.15,9.78,8.68,13.87,15.09,NaN
META,NaN,47.07,37.37,26.61,21.58,37.19,-1.12,15.68,21.94,22.17,NaN
MSFT,NaN,NaN,14.28,14.03,13.64,17.54,17.95,6.88,15.67,14.93,17.79
NFLX,NaN,32.39,35.07,27.68,24.01,18.80,6.46,6.64,15.66,15.85,NaN


In [19]:
check = annual.groupby("ticker")["fiscal_year"].agg(["min", "max", "count"])
check["gap"] = check["max"] - check["min"] + 1 - check["count"]

assert (check["gap"] == 0).all()
assert not annual.duplicated(["ticker", "fiscal_year"]).any()
check

,min,max,count,gap
ticker,,,,
AAPL,2016,2025,10,0
AMZN,2016,2025,10,0
GOOGL,2016,2025,10,0
META,2016,2025,10,0
MSFT,2017,2026,10,0
NFLX,2016,2025,10,0


In [20]:
annual.to_csv(out_dir / "faangm_annual_revenue.csv", index=False, encoding="utf-8-sig")
pivot.to_csv(out_dir / "faangm_annual_revenue_pivot.csv", encoding="utf-8-sig")
growth.to_csv(out_dir / "faangm_annual_revenue_growth.csv", encoding="utf-8-sig")